In [1]:
%pip install gradio

  Using cached gradio-6.27.0-py3-none-any.whl.metadata (17 kB)
  Using cached audioop_lts-0.2.2-cp313-abi3-win_amd64.whl.metadata (2.0 kB)
  Using cached brotli-1.2.0-cp313-cp313-win_amd64.whl.metadata (6.3 kB)
  Using cached fastapi-0.141.1-py3-none-any.whl.metadata (27 kB)
  Using cached gradio_client-2.7.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached hf_gradio-0.4.1-py3-none-any.whl.metadata (428 bytes)
  Using cached huggingface_hub-1.31.0-py3-none-any.whl.metadata (16 kB)
  Using cached orjson-3.12.0-cp313-cp313-win_amd64.whl.metadata (43 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.32-py3-none-any.whl.metadata (2.1 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
  Using cached starlette-1.6.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached typer-0.27.2

In [2]:
import gradio as gr
import joblib
import numpy as np
import dashscope
import json

# ========== 1. 加载模型（和之前一样） ==========
dashscope.api_key = ""
model = joblib.load("diabetes_model.pkl")
scaler = joblib.load("scaler.pkl")

FEATURE_NAMES = [
    "怀孕次数", "血糖浓度", "舒张压", "三头肌皮褶厚度",
    "2小时血清胰岛素", "BMI", "糖尿病遗传函数", "年龄"
]

# ========== 2. 沿用你刚才写好的 predict_patient 函数 ==========
def predict_patient(patient_dict):
    # ... 把你刚才 08_functions.ipynb 里的函数完整复制过来 ...
    # 注意：最后返回的是 result 字典
    # 1.1 把字典按顺序转成列表 → 二维数组
    patient_data = [patient_dict[name] for name in FEATURE_NAMES]
    patient_array = np.array([patient_data])
    
    # 1.2 标准化 + 预测概率
    patient_scaled = scaler.transform(patient_array)
    risk_prob = model.predict_proba(patient_scaled)[0, 1]
    
    # 1.3 调用 Qwen
    prompt = f"""
你是一名内分泌临床医师。请根据患者指标和模型预测概率，输出一段 JSON 格式的评估结果。
患者指标：{patient_dict}
模型预测患病概率：{risk_prob:.2%}

请严格按照以下 JSON 结构输出，不要包含任何其他文字：
{{
    "risk_level": "高风险/中风险/低风险",
    "main_factors": ["因素1", "因素2"],
    "suggestions": ["建议1", "建议2"],
    "disclaimer": "本结果仅供参考，不能替代临床就诊。"
}}
"""
    resp = dashscope.Generation.call(
        model="qwen-max",
        messages=[{"role": "user", "content": prompt}]
    )
    
    # 1.4 解析
    if resp.status_code == 200:
        try:
            result = json.loads(resp.output.text)
        except json.JSONDecodeError:
            result = {
                "risk_level": "解析失败",
                "main_factors": [],
                "suggestions": ["模型未按要求输出JSON，请检查Prompt"],
                "disclaimer": "本结果仅供参考"
            }
    else:
        result = {
            "risk_level": "调用失败",
            "main_factors": [],
            "suggestions": [resp.message],
            "disclaimer": "本结果仅供参考"
        }
    
    # 1.5 把概率也塞进结果字典（方便以后使用）
    result["prob"] = risk_prob
    return result

  # 把 pass 替换成你刚才写好的完整代码


# ========== 3. 包装给 Gradio 用的函数 ==========
def web_predict(pregnancies, glucose, blood_pressure, skin_thickness,
                insulin, bmi, pedigree, age):
    # 3.1 整理成字典
    patient = {
        "怀孕次数": pregnancies,
        "血糖浓度": glucose,
        "舒张压": blood_pressure,
        "三头肌皮褶厚度": skin_thickness,
        "2小时血清胰岛素": insulin,
        "BMI": bmi,
        "糖尿病遗传函数": pedigree,
        "年龄": age
    }
    
    # 3.2 调用你的黑盒子
    result = predict_patient(patient)
    
    # 3.3 返回给网页展示的内容（Markdown格式）
    report = f"""
## 糖尿病风险评估报告

**风险等级：** {result['risk_level']}
**模型概率：** {result['prob']:.2%}

**主要因素：** {'、'.join(result['main_factors']) if result['main_factors'] else '无'}

**生活建议：**
"""
    for i, s in enumerate(result["suggestions"], 1):
        report += f"{i}. {s}\n"
    
    report += f"\n---\n*{result['disclaimer']}*"
    return report


# ========== 4. 搭建网页界面 ==========
with gr.Blocks(title="糖尿病风险预测") as demo:
    gr.Markdown("# 🩺 糖尿病风险预测小工具")
    gr.Markdown("请调整下方滑块或输入数值，点击「开始评估」生成报告。")
    
    with gr.Row():
        with gr.Column():
            pregnancies = gr.Slider(0, 20, value=1, label="怀孕次数")
            glucose = gr.Slider(0, 200, value=100, label="血糖浓度")
            blood_pressure = gr.Slider(0, 120, value=70, label="舒张压")
            skin_thickness = gr.Slider(0, 100, value=20, label="三头肌皮褶厚度")
            insulin = gr.Slider(0, 900, value=80, label="2小时血清胰岛素")
            bmi = gr.Slider(0, 70, value=25, label="BMI")
            pedigree = gr.Slider(0, 3, value=0.5, label="糖尿病遗传函数")
            age = gr.Slider(0, 100, value=30, label="年龄")
            
            btn = gr.Button("开始评估", variant="primary")
        
        with gr.Column():
            output = gr.Markdown(label="评估报告")
    
    btn.click(
        fn=web_predict,
        inputs=[pregnancies, glucose, blood_pressure, skin_thickness,
                insulin, bmi, pedigree, age],
        outputs=output
    )

# 启动网页
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
import sys
print(sys.executable)  # 看看当前到底用的哪个 Python
import gradio as gr
print(gr.__version__)  # 看看能不能打印出版本号

D:\Anaconda3\python.exe
6.27.0
